# Tracing： LangSmith 和 Langfuse

上一节讲 OpenTelemetry GenAI 语义约定——span 怎么命名、属性怎么写、内容怎么外存。本节讲你实际打开的两个产品面：LangSmith（LangChain 生态原生）和 Langfuse（开源、可自托管）。约定解决「跨后端同质」；产品解决「看轨迹、打分、建数据集、回归」。

## 问题描述

Agent 跑 40 步失败在第 38 步。日志告诉你「search 超时」，对不上第几轮 Thought、哪个子 agent、花了多少 token。没有轨迹树，你只能重跑碰运气。LangSmith 和 Langfuse 把一次 invoke 收成可点开的 run 树：根是请求，子节点是 LLM / tool / chain，边上挂 latency、token、输入输出（或引用）。缺的是把探针接到产品，而不是再发明一套 span 命名。

## 基本概念

### Run 树，不是平铺日志

一次用户请求 = 一条 trace / 一个根 run。子 run 是模型调用、工具调用、子图。UI 靠父子关系还原循环；导出时要能断言「每个 tool run 的 parent 是当前 agent/chain，不是孤儿」。这和上一节的 span 父子链接是同一件事，只是产品把树画出来并接上筛选、对比、回放。

### LangSmith：LangChain 原生观测 + 评估

|能力|作用|
|---|---|
|Tracing|LangChain/LangGraph 自动挂 callback，一键看 run 树|
|Datasets|把生产失败轨迹抽成固定输入集|
|Evaluation|规则 / LLM-as-judge / 人工反馈打分|
|Playground|改提示词后对照同一 dataset 回归|

适合已经在 LangChain/LangGraph 上的团队：探针几乎零配置，评估闭环和轨迹在同一产品里。付费云为主；企业可自托管，但心智默认是「LangChain 的观测层」。

### Langfuse：开源观测 + Prompt / Score

|能力|作用|
|---|---|
|Tracing|SDK / OpenTelemetry / 框架集成，自托管或云|
|Scores|数值或分类分数挂到 trace / observation|
|Prompts|版本化提示词，和轨迹对照|
|Datasets / Experiments|批量跑评，对比版本|

适合要自托管、多框架（不只 LangChain）、或要把轨迹接到自己的评估管道的团队。心智是「开源 LLMOps」：你拥有数据，Schema 可对齐 OTel GenAI，后端自己选。

### 怎么选（机制，不是品牌）

- 全栈已是 LangChain/LangGraph，要最快看到树 + 内置评测 → LangSmith。
- 要自托管、多框架、或轨迹必须进自己的仓库/评测 → Langfuse（或 OTel → 自建后端）。
- 两者都能做：打分、数据集、失败回放。差在集成摩擦和数据主权，不在「能不能画树」。

### 探针接到产品

编码可演示的三件事：
1. 包一层 handler / SDK，让一次 agent invoke 产出带父子关系的 run 树。
2. 给失败轨迹打 score（规则或人工），再抽进 dataset。
3. 改提示词后对同一 dataset 重跑，对比 score 分布——不是只看单次成功与否。

## 什么时候这种模式出错

- 轨迹里默认塞全文提示词与工具返回。PII 和密钥进 SaaS；上一节的外存 + 引用规则在产品层同样适用。
- 只开根 span，子 LLM/tool 没挂上。UI 里是一条扁线，第 38 步仍对不上。
- 评估分数没有稳定标签。今天 `correctness=0.8`，明天换 judge 提示词，回归不可比。
- 把产品当唯一真相源又不导出。换供应商或自建时，历史轨迹锁死在专有格式里。
- 为了好看仪表盘采样掉失败路径。产品关心的是尾部失败，不是平均 latency。


# 开始编码

对应本章核心：**Run 树（父子链接）**、**Score → Dataset**、**同 dataset 提示词回归对比**、**默认内容引用不落全文**。  
先做内存版「产品面」探针（LangSmith / Langfuse 同构机制）；再用 **LangChain Callback + DeepSeek** 跑真实 agent（不硬凑 PyTorch）。不强制云端 API：无 `LANGSMITH_*` / `LANGFUSE_*` 时本地树仍可断言；无 `DEEPSEEK_API_KEY` 则生产示例 SKIP。


## 1. 教学玩具：产品面 Run 树 + Score + Dataset

- **Run 树**：根 `agent`，子 `llm` / `tool`；可断言 parent。
- **内容**：默认只存 `input_ref` / `output_ref`。
- **Score**：稳定标签（如 `rule.correctness`）挂到 trace。
- **Dataset / Experiment**：失败轨迹抽样本；两版提示词同集重跑比分。


In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import time
import uuid
from dataclasses import dataclass, field
from typing import Any, Callable, Literal


@dataclass
class ContentStore:
    """外存：run 属性只留引用。"""

    blobs: dict[str, str] = field(default_factory=dict)

    def put(self, text: str) -> str:
        """
        Args:
            text: 完整输入/输出。

        Returns:
            ref: 内容引用 ID。
        """
        ref = f"cnt_{uuid.uuid4().hex[:10]}"
        self.blobs[ref] = text
        return ref

    def get(self, ref: str) -> str | None:
        return self.blobs.get(ref)


@dataclass
class Run:
    """一次 observation / run（对齐 LangSmith run / Langfuse observation）。"""

    id: str
    name: str
    kind: Literal["agent", "llm", "tool", "chain"]
    parent_id: str | None
    trace_id: str
    attrs: dict[str, Any] = field(default_factory=dict)
    input_ref: str | None = None
    output_ref: str | None = None
    error: str | None = None
    started_ms: float = 0.0
    ended_ms: float = 0.0

    @property
    def latency_ms(self) -> float:
        return max(0.0, self.ended_ms - self.started_ms)


@dataclass
class Score:
    """挂到 trace 的稳定标签分数（Langfuse score / LangSmith feedback）。"""

    name: str
    value: float
    trace_id: str
    comment: str = ""


@dataclass
class DatasetItem:
    """从轨迹抽出的固定输入样本。"""

    id: str
    input_text: str
    expected: str | None = None
    source_trace_id: str = ""


class ProductTracer:
    """内存版产品面：run 树 + score + dataset + experiment。"""

    def __init__(self, *, capture_content: bool = False, store: ContentStore | None = None) -> None:
        self.capture_content = capture_content
        self.store = store or ContentStore()
        self.runs: list[Run] = []
        self.scores: list[Score] = []
        self.datasets: dict[str, list[DatasetItem]] = {}
        self._stack: list[str] = []
        self._trace_id: str | None = None

    def clear(self) -> None:
        self.runs.clear()
        self.scores.clear()
        self._stack.clear()
        self._trace_id = None

    def _attach(self, text: str | None) -> tuple[str | None, str | None]:
        if text is None:
            return None, None
        ref = self.store.put(text)
        raw = text[:2000] if self.capture_content else None
        return ref, raw

    def start_run(
        self,
        *,
        name: str,
        kind: Literal["agent", "llm", "tool", "chain"],
        input_text: str | None = None,
        attrs: dict[str, Any] | None = None,
    ) -> Run:
        """
        Args:
            name: run 名称。
            kind: agent / llm / tool / chain。
            input_text: 可选输入（默认只存引用）。
            attrs: 额外属性。

        Returns:
            run: 已入栈的 run。
        """
        if self._trace_id is None:
            self._trace_id = f"tr_{uuid.uuid4().hex[:12]}"
        parent = self._stack[-1] if self._stack else None
        ref, raw = self._attach(input_text)
        run = Run(
            id=f"run_{uuid.uuid4().hex[:12]}",
            name=name,
            kind=kind,
            parent_id=parent,
            trace_id=self._trace_id,
            attrs=dict(attrs or {}),
            input_ref=ref,
            started_ms=time.time() * 1000,
        )
        if raw is not None:
            run.attrs["input_preview"] = raw
        self.runs.append(run)
        self._stack.append(run.id)
        return run

    def end_run(self, run: Run, *, output_text: str | None = None, error: str | None = None) -> None:
        """
        Args:
            run: start_run 返回的对象。
            output_text: 输出（默认只存引用）。
            error: 失败信息。
        """
        ref, raw = self._attach(output_text)
        run.output_ref = ref
        run.error = error
        run.ended_ms = time.time() * 1000
        if raw is not None:
            run.attrs["output_preview"] = raw
        if self._stack and self._stack[-1] == run.id:
            self._stack.pop()
        if not self._stack:
            self._trace_id = None

    def run(
        self,
        *,
        name: str,
        kind: Literal["agent", "llm", "tool", "chain"],
        fn: Callable[[], str],
        input_text: str | None = None,
        attrs: dict[str, Any] | None = None,
    ) -> str:
        """
        上下文风格：start → fn → end，保证父子链接。

        Returns:
            output: fn 返回值。
        """
        r = self.start_run(name=name, kind=kind, input_text=input_text, attrs=attrs)
        try:
            out = fn()
            self.end_run(r, output_text=out)
            return out
        except Exception as e:
            self.end_run(r, error=str(e))
            raise

    def score(self, *, name: str, value: float, trace_id: str, comment: str = "") -> Score:
        """
        Args:
            name: 稳定标签，如 ``rule.correctness``。
            value: 数值分。
            trace_id: 所属 trace。
            comment: 备注。

        Returns:
            score: 已记录分数。
        """
        s = Score(name=name, value=float(value), trace_id=trace_id, comment=comment)
        self.scores.append(s)
        return s

    def add_to_dataset(
        self,
        dataset: str,
        *,
        input_text: str,
        expected: str | None = None,
        source_trace_id: str = "",
    ) -> DatasetItem:
        """
        从失败/关注轨迹抽固定样本。

        Returns:
            item: dataset 条目。
        """
        item = DatasetItem(
            id=f"ds_{uuid.uuid4().hex[:10]}",
            input_text=input_text,
            expected=expected,
            source_trace_id=source_trace_id,
        )
        self.datasets.setdefault(dataset, []).append(item)
        return item

    def export_tree(self, trace_id: str | None = None) -> list[dict[str, Any]]:
        """
        导出可移植 JSON（避免锁死在单一 SaaS）。

        Returns:
            rows: run 列表（无全文）。
        """
        rows = []
        for r in self.runs:
            if trace_id and r.trace_id != trace_id:
                continue
            rows.append(
                {
                    "id": r.id,
                    "name": r.name,
                    "kind": r.kind,
                    "parent_id": r.parent_id,
                    "trace_id": r.trace_id,
                    "input_ref": r.input_ref,
                    "output_ref": r.output_ref,
                    "error": r.error,
                    "latency_ms": round(r.latency_ms, 2),
                    "attrs": {k: v for k, v in r.attrs.items() if not k.endswith("_preview")},
                }
            )
        return rows

    def children_of(self, run_id: str) -> list[Run]:
        return [r for r in self.runs if r.parent_id == run_id]


def rule_correctness(output: str, *, must_contain: str) -> float:
    """
    稳定规则分：输出是否包含关键子串。

    Returns:
        score: 1.0 或 0.0。
    """
    return 1.0 if must_contain.lower() in output.lower() else 0.0


print("ProductTracer ready | run-tree + score + dataset")


## 2. 玩具示例：父子树、引用、打分、两版提示词回归


In [ ]:
def demo_product_tracing() -> None:
    """断言产品面机制：树、引用、score、dataset 回归。"""
    tr = ProductTracer(capture_content=False)
    secret = "SSN 123-45-6789 order 88991"

    def agent_body() -> str:
        thought = tr.run(
            name="chat deepseek-v4-flash",
            kind="llm",
            input_text=secret,
            attrs={"provider": "deepseek", "model": "deepseek-v4-flash"},
            fn=lambda: "need tool",
        )
        tool_out = tr.run(
            name="execute_tool lookup_order",
            kind="tool",
            input_text='{"order_id":"88991"}',
            attrs={"tool": "lookup_order"},
            fn=lambda: '{"status":"paid"}',
        )
        return f"{thought}; {tool_out}"

    out = tr.run(
        name="invoke_agent triage",
        kind="agent",
        input_text=secret,
        attrs={"agent": "triage", "backend": "langsmith-like"},
        fn=agent_body,
    )
    assert "paid" in out

    tree = tr.export_tree()
    by_name = {r["name"]: r for r in tree}
    agent = by_name["invoke_agent triage"]
    llm = by_name["chat deepseek-v4-flash"]
    tool = by_name["execute_tool lookup_order"]
    assert llm["parent_id"] == agent["id"]
    assert tool["parent_id"] == agent["id"]
    assert secret not in json.dumps(tree, ensure_ascii=False)
    assert tr.store.get(agent["input_ref"]) == secret
    print("run-tree + content-by-ref ok")

    # 稳定标签打分 → 抽进 dataset
    tid = agent["trace_id"]
    s = tr.score(name="rule.correctness", value=rule_correctness(out, must_contain="paid"), trace_id=tid)
    assert s.value == 1.0
    tr.add_to_dataset("triage_fail", input_text="查订单 88991", expected="paid", source_trace_id=tid)
    assert len(tr.datasets["triage_fail"]) == 1
    print("score + dataset ok")

    # 同 dataset：弱提示词 vs 强提示词 → 分数分布可比
    items = [
        DatasetItem(id="1", input_text="查订单 88991", expected="paid"),
        DatasetItem(id="2", input_text="订单 10001 状态", expected="paid"),
        DatasetItem(id="3", input_text="随便聊聊天气", expected=None),
    ]

    def run_prompt(version: str, item: DatasetItem) -> float:
        def body() -> str:
            if "订单" in item.input_text or "order" in item.input_text.lower():
                if version == "weak":
                    # 弱版：常漏掉 status
                    return "已处理"
                return tr.run(
                    name="execute_tool lookup_order",
                    kind="tool",
                    fn=lambda: "status=paid",
                )
            return "今天晴"

        out2 = tr.run(
            name=f"invoke_agent {version}",
            kind="agent",
            input_text=item.input_text,
            attrs={"prompt_version": version},
            fn=body,
        )
        if item.expected is None:
            return 1.0 if "晴" in out2 else 0.0
        return rule_correctness(out2, must_contain=item.expected)

    weak = [run_prompt("weak", it) for it in items]
    strong = [run_prompt("strong", it) for it in items]
    assert sum(strong) > sum(weak)
    print(f"experiment weak={weak} strong={strong}")
    print("TOY DEMO OK")


demo_product_tracing()


## 3. 生产级：LangChain Callback 树 + DeepSeek

用 `BaseCallbackHandler` 把真实 LLM/tool 事件写进 `ProductTracer`（LangSmith callback / Langfuse 观测同构）。可选：若环境有 `LANGCHAIN_TRACING_V2` + API key，同时打开官方 LangSmith；本节断言以本地导出树为准。需 `DEEPSEEK_API_KEY`。


In [ ]:
import os
import sys
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.outputs import LLMResult
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROD_TR = ProductTracer(capture_content=False)
LAST_TRACE_ID = ""


class ProductCallbackHandler(BaseCallbackHandler):
    """把 LangChain 事件映射成产品面 run 树。"""

    def __init__(self, tracer: ProductTracer) -> None:
        self.tracer = tracer
        self._llm_runs: dict[str, Run] = {}
        self._tool_runs: dict[str, Run] = {}
        self._agent_run: Run | None = None

    def on_chain_start(self, serialized: dict[str, Any], inputs: dict[str, Any], *, run_id, **kwargs: Any) -> None:
        name = (serialized or {}).get("name") or kwargs.get("name") or "chain"
        # 根 agent / graph 入口
        if self._agent_run is None and name in {"LangGraph", "agent", "RunnableSequence"}:
            text = ""
            msgs = inputs.get("messages") if isinstance(inputs, dict) else None
            if msgs:
                last = msgs[-1]
                text = getattr(last, "content", str(last))
            elif isinstance(inputs, dict) and "input" in inputs:
                text = str(inputs["input"])
            self._agent_run = self.tracer.start_run(
                name="invoke_agent triage",
                kind="agent",
                input_text=str(text) or None,
                attrs={"framework": "langchain", "backend": "product-callback"},
            )

    def on_chain_end(self, outputs: dict[str, Any], *, run_id, **kwargs: Any) -> None:
        if self._agent_run is not None and not self.tracer._stack:
            return
        # 仅在 agent run 仍在栈顶相关时结束：由显式 finish_agent 处理更稳
        return

    def on_chat_model_start(self, serialized: dict[str, Any], messages: list, *, run_id, **kwargs: Any) -> None:
        flat = []
        for batch in messages:
            for m in batch:
                flat.append(f"{getattr(m, 'type', type(m).__name__)}:{getattr(m, 'content', m)}")
        r = self.tracer.start_run(
            name=f"chat {MODEL}",
            kind="llm",
            input_text="\n".join(flat)[:4000],
            attrs={"provider": "deepseek", "model": MODEL},
        )
        self._llm_runs[str(run_id)] = r

    def on_llm_end(self, response: LLMResult, *, run_id, **kwargs: Any) -> None:
        r = self._llm_runs.pop(str(run_id), None)
        if r is None:
            return
        text = ""
        try:
            text = response.generations[0][0].text
        except Exception:
            text = str(response)
        self.tracer.end_run(r, output_text=text)

    def on_llm_error(self, error: BaseException, *, run_id, **kwargs: Any) -> None:
        r = self._llm_runs.pop(str(run_id), None)
        if r is not None:
            self.tracer.end_run(r, error=str(error))

    def on_tool_start(self, serialized: dict[str, Any], input_str: str, *, run_id, **kwargs: Any) -> None:
        name = (serialized or {}).get("name") or kwargs.get("name") or "tool"
        r = self.tracer.start_run(
            name=f"execute_tool {name}",
            kind="tool",
            input_text=str(input_str),
            attrs={"tool": name},
        )
        self._tool_runs[str(run_id)] = r

    def on_tool_end(self, output: Any, *, run_id, **kwargs: Any) -> None:
        r = self._tool_runs.pop(str(run_id), None)
        if r is not None:
            self.tracer.end_run(r, output_text=str(output))

    def on_tool_error(self, error: BaseException, *, run_id, **kwargs: Any) -> None:
        r = self._tool_runs.pop(str(run_id), None)
        if r is not None:
            self.tracer.end_run(r, error=str(error))

    def finish_agent(self, output_text: str) -> str:
        """
        显式关闭根 agent run。

        Returns:
            trace_id: 本轮 trace。
        """
        if self._agent_run is not None:
            # 若 LLM/tool 异常导致栈残留，先清空到 agent
            while self.tracer._stack and self.tracer._stack[-1] != self._agent_run.id:
                orphan = next(r for r in self.tracer.runs if r.id == self.tracer._stack[-1])
                self.tracer.end_run(orphan, error="unclosed")
            self.tracer.end_run(self._agent_run, output_text=output_text)
            tid = self._agent_run.trace_id
            self._agent_run = None
            return tid
        return ""


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def lookup_order_impl(order_id: str) -> str:
    return json.dumps({"order_id": order_id, "status": "paid", "amount": 99}, ensure_ascii=False)


class OrderArgs(BaseModel):
    order_id: str = Field(description="订单号数字")


LOOKUP_TOOL = StructuredTool.from_function(
    name="lookup_order",
    description="Lookup order status by id.",
    func=lambda order_id: lookup_order_impl(order_id),
    args_schema=OrderArgs,
)


def build_triage_agent(*, system: str) -> Any:
    """
    Args:
        system: 系统提示词（用于 experiment 对比）。

    Returns:
        agent: LangChain create_agent。
    """
    return create_agent(get_llm(), [LOOKUP_TOOL], system_prompt=system)


WEAK_SYSTEM = "你是客服。尽量简短回复。可以调用 lookup_order。"
STRONG_SYSTEM = (
    "你是订单客服。用户提到订单号时必须调用 lookup_order，"
    "最终回复必须包含英文 status 值（如 paid）。中文简述，<=40字。"
)


def run_traced_agent_impl(user_text: str, *, system: str = STRONG_SYSTEM) -> str:
    """
    一轮带 callback 树的 agent。

    Returns:
        json: output + tree + optional score。
    """
    global LAST_TRACE_ID, PROD_TR
    PROD_TR.clear()
    handler = ProductCallbackHandler(PROD_TR)
    # 手动开根，避免不同 create_agent 版本 chain 名不一致
    root = PROD_TR.start_run(
        name="invoke_agent triage",
        kind="agent",
        input_text=user_text,
        attrs={"framework": "langchain", "prompt_hash": hashlib.md5(system.encode()).hexdigest()[:8]},
    )
    handler._agent_run = root

    agent = build_triage_agent(system=system)
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_text)]},
        config={"callbacks": [handler]},
    )
    messages = result.get("messages") or []
    final = ""
    for m in reversed(messages):
        if isinstance(m, AIMessage) and m.content and not m.tool_calls:
            final = str(m.content)
            break
    if not final and messages:
        final = str(getattr(messages[-1], "content", messages[-1]))

    # 关闭未结束的子 run，再关根
    while PROD_TR._stack and PROD_TR._stack[-1] != root.id:
        orphan = next(r for r in PROD_TR.runs if r.id == PROD_TR._stack[-1])
        PROD_TR.end_run(orphan, error="unclosed")
    PROD_TR.end_run(root, output_text=final)
    LAST_TRACE_ID = root.trace_id
    handler._agent_run = None

    sc = rule_correctness(final, must_contain="paid") if any(ch.isdigit() for ch in user_text) else None
    if sc is not None:
        PROD_TR.score(name="rule.correctness", value=sc, trace_id=LAST_TRACE_ID, comment="stable rule")
        if sc < 1.0:
            PROD_TR.add_to_dataset(
                "prod_regress",
                input_text=user_text,
                expected="paid",
                source_trace_id=LAST_TRACE_ID,
            )

    return json.dumps(
        {
            "output": final,
            "trace_id": LAST_TRACE_ID,
            "score": sc,
            "tree": PROD_TR.export_tree(LAST_TRACE_ID),
            "langsmith_env": bool(os.getenv("LANGCHAIN_API_KEY") or os.getenv("LANGSMITH_API_KEY")),
        },
        ensure_ascii=False,
    )


def export_tree_impl() -> str:
    """
    Returns:
        json: 可移植 run 树（无全文）。
    """
    return json.dumps({"runs": PROD_TR.export_tree(), "scores": [s.__dict__ for s in PROD_TR.scores]}, ensure_ascii=False)


def fetch_content_impl(ref: str) -> str:
    text = PROD_TR.store.get(ref)
    if text is None:
        return json.dumps({"error": "missing", "ref": ref})
    return json.dumps({"ref": ref, "text": text}, ensure_ascii=False)


def experiment_impl(dataset_name: str = "builtin") -> str:
    """
    同 dataset 上 weak vs strong 提示词回归。

    Returns:
        json: 两版分数分布。
    """
    if dataset_name == "builtin" or dataset_name not in PROD_TR.datasets:
        items = [
            DatasetItem(id="a", input_text="帮我查订单 88991", expected="paid"),
            DatasetItem(id="b", input_text="订单10001付了吗", expected="paid"),
        ]
    else:
        items = PROD_TR.datasets[dataset_name]

    def eval_version(system: str, label: str) -> list[float]:
        scores: list[float] = []
        for it in items:
            payload = json.loads(run_traced_agent_impl(it.input_text, system=system))
            val = payload.get("score")
            if val is None:
                val = rule_correctness(payload["output"], must_contain=it.expected or "")
            scores.append(float(val))
            # 给 experiment 打版本标签分
            PROD_TR.score(
                name=f"experiment.{label}",
                value=float(val),
                trace_id=payload["trace_id"],
                comment=it.id,
            )
        return scores

    weak_scores = eval_version(WEAK_SYSTEM, "weak")
    strong_scores = eval_version(STRONG_SYSTEM, "strong")
    return json.dumps(
        {
            "dataset_size": len(items),
            "weak": weak_scores,
            "strong": strong_scores,
            "weak_mean": sum(weak_scores) / len(weak_scores),
            "strong_mean": sum(strong_scores) / len(strong_scores),
        },
        ensure_ascii=False,
    )


class RunArgs(BaseModel):
    text: str
    strong: bool = True


class ExpArgs(BaseModel):
    dataset_name: str = "builtin"


class FetchArgs(BaseModel):
    ref: str


class EmptyArgs(BaseModel):
    pass


def build_control_tools() -> list[StructuredTool]:
    def _run(**kwargs: Any) -> str:
        a = RunArgs(**kwargs)
        return run_traced_agent_impl(a.text, system=STRONG_SYSTEM if a.strong else WEAK_SYSTEM)

    def _export(**kwargs: Any) -> str:
        return export_tree_impl()

    def _fetch(**kwargs: Any) -> str:
        return fetch_content_impl(FetchArgs(**kwargs).ref)

    def _exp(**kwargs: Any) -> str:
        return experiment_impl(ExpArgs(**kwargs).dataset_name)

    return [
        StructuredTool.from_function(
            name="run_traced_agent",
            description="Run triage agent with product run-tree callback.",
            func=_run,
            args_schema=RunArgs,
        ),
        StructuredTool.from_function(
            name="export_tree",
            description="Export portable run tree + scores without full prompts.",
            func=_export,
            args_schema=EmptyArgs,
        ),
        StructuredTool.from_function(
            name="fetch_content",
            description="Fetch full text by content ref.",
            func=_fetch,
            args_schema=FetchArgs,
        ),
        StructuredTool.from_function(
            name="run_experiment",
            description="Compare weak vs strong prompts on same dataset.",
            func=_exp,
            args_schema=ExpArgs,
        ),
    ]


CONTROL_TOOLS = build_control_tools()


def build_control_agent() -> Any:
    system = (
        "You operate a tracing control plane (LangSmith/Langfuse-like).\\n"
        "Flow: run_traced_agent -> export_tree -> optionally fetch_content / run_experiment.\\n"
        "Explain parent links and that prompts are refs-only. Chinese."
    )
    return create_agent(get_llm(), CONTROL_TOOLS, system_prompt=system)


def format_agent_messages(messages: list[BaseMessage]) -> str:
    lines: list[str] = []
    for m in messages:
        if isinstance(m, HumanMessage):
            lines.append(f"USER: {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    lines.append(f"ACTION: {tc['name']}({tc.get('args') or {}})")
            if m.content:
                lines.append(f"ASSISTANT: {m.content}")
        elif isinstance(m, ToolMessage):
            content = m.content if len(str(m.content)) < 600 else str(m.content)[:600] + "..."
            lines.append(f"OBS[{m.name}]: {content}")
    return "\n".join(lines)


print(f"LangChain product tracing ready | {MODEL}")


## 4. 生产示例：追踪一轮 Agent + 同集回归

无 `DEEPSEEK_API_KEY` 则 SKIP。


In [ ]:
def demo_production_tracing() -> None:
    """生产：run 树父子 + 可选 experiment；无 key 则 SKIP。"""
    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP production: DEEPSEEK_API_KEY missing")
        return

    raw = run_traced_agent_impl("帮我查一下订单 88991 的状态", system=STRONG_SYSTEM)
    payload = json.loads(raw)
    print("=== traced run ===")
    print(json.dumps(payload, ensure_ascii=False, indent=2)[:1200])
    tree = payload["tree"]
    agents = [r for r in tree if r["kind"] == "agent"]
    kids = [r for r in tree if r["parent_id"] == agents[0]["id"]]
    assert agents, "missing agent root"
    assert any(r["kind"] == "llm" for r in kids), "llm must be child of agent"
    # tool 可能出现（强提示词应调用）
    kinds = {r["kind"] for r in kids}
    print("child kinds under agent:", kinds)
    assert "paid" in payload["output"].lower() or payload.get("score") == 1.0 or any(
        r["kind"] == "tool" for r in tree
    )
    dumped = json.dumps(tree, ensure_ascii=False)
    assert "SSN" not in dumped
    print("parent links ok; no full secret in export")

    exp = json.loads(experiment_impl("builtin"))
    print("=== experiment weak vs strong ===")
    print(exp)
    assert exp["strong_mean"] >= exp["weak_mean"]
    print("PROD DEMO OK")


demo_production_tracing()
